# Week 10 Coding Practice: Can We Trust an Explanation?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/main/module10/week10_explainability_practice.ipynb)

**Estimated time:** 45-60 minutes
**Format:** ungraded guided practice

A model can predict reasonably well and still be explained in ways that mislead. In this activity you will evaluate a trained regression model, generate a global explanation (permutation feature importance) and a local explanation (a SHAP waterfall plot) for one prediction, then challenge that explanation using correlated features. The guiding question throughout is:

> A convincing explanation does not establish that a prediction is correct or reliable. What evidence would it take to trust one?


## Learning objectives

By the end of this activity, you should be able to:

1. evaluate a trained regression model's predictive performance using MAE and R²;
2. compute and interpret permutation feature importance on held-out data;
3. generate and interpret a SHAP waterfall plot for one individual prediction;
4. show how correlated features can distort a permutation-importance-based explanation;
5. write a short, evidence-based assessment of a model's reliability that distinguishes what an explanation shows from whether a prediction is correct.


## 1. Understand the prediction problem

We use the California housing dataset, bundled with scikit-learn. Each row is a California census block group in the 1990 census. The task: predict `MedHouseVal`, the median house value in that block group (in units of \$100,000), from eight numeric predictors.

### Variables used in this activity

| Variable | Role | Meaning |
| --- | --- | --- |
| `MedHouseVal` | response | median house value, \$100,000s |
| `MedInc` | numeric predictor | median household income, \$10,000s |
| `HouseAge` | numeric predictor | median house age, years |
| `AveRooms` | numeric predictor | average rooms per household |
| `AveBedrms` | numeric predictor | average bedrooms per household |
| `Population` | numeric predictor | block group population |
| `AveOccup` | numeric predictor | average household occupancy |
| `Latitude` | numeric predictor | block group latitude |
| `Longitude` | numeric predictor | block group longitude |

The model below is already fit for you. Building and tuning the regression model is not the point of this activity (you practiced that in Module 4 and Module 5); evaluating, explaining, and challenging an explanation of an already-fit model is.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.inspection import permutation_importance
import shap

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42

housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

model = RandomForestRegressor(
    n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
)
model.fit(X_train, y_train)

print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}")
print(f"Predictors: {list(X.columns)}")


### Reasoning checkpoint

Before looking at any performance numbers, write one or two sentences: do you expect a model trained on these eight predictors to predict median house value almost exactly, approximately, or poorly? What makes you expect that?


## 2. Evaluate predictive performance

Compute the model's performance on the **test** set only. Report both MAE (mean absolute error, in the same \$100,000 units as the target) and R² (the proportion of variance explained).


In [ ]:
test_predictions = model.predict(X_test)

# TODO 1: Compute the test MAE and test R^2 using mean_absolute_error and r2_score.
# Store them as test_mae and test_r2.
test_mae = ...
test_r2 = ...

print(f"Test MAE: {test_mae:.3f} (~${test_mae * 100_000:,.0f})")
print(f"Test R^2: {test_r2:.3f}")


### Reasoning checkpoint: is this model good enough to trust?

Using the MAE and R² you just computed:

- In plain words, how far off is a typical prediction, in dollars?
- Is an R² at this level closer to "the model captures most of the pattern" or "the model captures a modest part of the pattern"?
- Could a model with this level of performance still be useful for some purposes but not others? Name one purpose it would be unsuitable for.


## 3. Global explanation: permutation feature importance

Permutation importance asks: if we destroy one feature's relationship with the target by shuffling its values, how much worse does the model get? A larger drop in performance means the model relied on that feature more. This is computed on **held-out test data**, not training data, so it reflects something about generalization rather than overfitting to one feature.


In [ ]:
# TODO 2: Compute permutation importance of each predictor using the test set.
# Use permutation_importance(model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE).
perm_result = ...

importance_table = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": perm_result.importances_mean,
        "importance_std": perm_result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

display(importance_table.round(3))

fig, ax = plt.subplots(figsize=(7, 4.5))
importance_table.sort_values("importance_mean").plot.barh(
    x="feature", y="importance_mean", xerr="importance_std",
    ax=ax, legend=False, color="#0072B2",
)
ax.set_xlabel("Mean decrease in R² when shuffled")
ax.set_ylabel("Predictor")
ax.set_title("Permutation feature importance (test set)")
plt.tight_layout()
plt.show()


### Reasoning checkpoint: what this ranking does and does not establish

In Module 5 you learned that tree-based (impurity) feature importance does not prove causation and can shift with correlated predictors. Permutation importance is a different method, computed on held-out data instead of training splits, but the same caution applies:

- Which feature ranks highest? Does a high rank mean that feature *causes* higher house values?
- If two predictors were nearly identical copies of each other, what would you expect to happen to each one's individual importance score, even if the pair together matters a lot?
- Name one thing permutation importance tells you about this model's behavior, and one thing it cannot tell you about California housing prices in general.


## 4. Identify a prediction with a large error

Before explaining a *specific* prediction, we need one worth explaining: a case where the model is clearly wrong, not a case where it is already correct.

One detail about this dataset: about 5% of block groups have a median house value capped at the dataset's maximum recorded value (\$500,001), which is a data-collection artifact, not something the model could ever predict exactly. We exclude those capped rows below so the "largest error" we pick reflects a genuine model miss, not a label ceiling the model cannot see past.


In [ ]:
residuals = pd.DataFrame(
    {"actual": y_test.values, "predicted": test_predictions}, index=y_test.index
)
residuals["abs_error"] = (residuals["actual"] - residuals["predicted"]).abs()

# Exclude rows where the actual value is capped at the dataset maximum (not a real model error).
uncapped = residuals[residuals["actual"] < 5.0]

# TODO 3: Find the index (row label) of the uncapped row with the largest absolute error.
# Store it as worst_idx.
worst_idx = ...

worst_row = residuals.loc[worst_idx]
print(f"Selected row index: {worst_idx}")
print(
    f"Actual: ${worst_row['actual'] * 100_000:,.0f}   "
    f"Predicted: ${worst_row['predicted'] * 100_000:,.0f}   "
    f"Error: ${worst_row['abs_error'] * 100_000:,.0f}"
)
display(X_test.loc[[worst_idx]])


### Actual vs. predicted comparison

Before moving on, write one sentence stating the actual value, the predicted value, and the dollar gap between them, in plain language suitable for someone who has not seen this notebook.


## 5. Explain that prediction with SHAP

SHAP assigns each predictor a contribution (positive or negative) that explains how the model moved from its average prediction (the "base value") to its specific prediction for this one row. We use `shap.TreeExplainer`, which works efficiently with tree-based models like our random forest.


In [ ]:
explainer = shap.TreeExplainer(model)
row_to_explain = X_test.loc[[worst_idx]]
shap_explanation = explainer(row_to_explain)

shap.plots.waterfall(shap_explanation[0], show=False)
fig = plt.gcf()
fig.set_size_inches(8, 5)
plt.title(f"SHAP explanation for row {worst_idx}")
plt.tight_layout()
plt.show()


### Reasoning checkpoint: what the waterfall plot shows

- What is the base value (the model's average prediction before seeing this row's features), and what is the final predicted value for this row?
- Name one feature that pushed the prediction up and one that pushed it down. By roughly how much each, in dollars?
- The model's prediction for this row was still far from the actual value (Step 4). Does the SHAP plot explain *why the model's prediction is wrong*, or does it only explain *how the model arrived at its own prediction*? These are not the same question. Write one sentence distinguishing them.


## 6. Challenge the explanation: correlated features

`Latitude` and `Longitude` are strongly correlated with each other (every California location has both). Standard permutation importance shuffles one feature at a time, which can create unrealistic (latitude, longitude) combinations that do not correspond to any real place. That can distort how much importance each one appears to carry, individually, compared to permuting them *together* as a pair, which keeps each shuffled row's location internally consistent.


In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
baseline_r2 = r2_score(y_test, model.predict(X_test))

def permuted_r2_drop(columns, n_repeats=10):
    # Mean R^2 drop from shuffling `columns` together, `n_repeats` times.
    drops = []
    for _ in range(n_repeats):
        X_shuffled = X_test.copy()
        shuffle_order = rng.permutation(len(X_shuffled))
        for column in columns:
            X_shuffled[column] = X_shuffled[column].values[shuffle_order]
        drops.append(baseline_r2 - r2_score(y_test, model.predict(X_shuffled)))
    return np.mean(drops)

# TODO 4: Compute the R^2 drop for shuffling Latitude alone, Longitude alone,
# and Latitude+Longitude together (as one joint shuffle). Store the three
# results as lat_drop, lon_drop, and joint_drop.
lat_drop = ...
lon_drop = ...
joint_drop = ...

comparison = pd.DataFrame(
    {
        "Shuffled independently (sum)": [lat_drop + lon_drop],
        "Shuffled together (joint)": [joint_drop],
    },
    index=["R² drop"],
)
display(comparison.round(3))

fig, ax = plt.subplots(figsize=(6, 4))
comparison.T.plot.bar(ax=ax, legend=False, color="#D55E00")
ax.set_ylabel("R² drop")
ax.set_title("Latitude + Longitude: independent sum vs. joint permutation")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


### Quick Check: critique an AI claim

Suppose an AI assistant looked only at the individual permutation importance values from Step 3 and wrote:

> "Longitude has a lower importance score than Latitude, so the model relies on Latitude for location information and Longitude contributes comparatively little."

Using the comparison you just computed, write a correction. Does the gap between the independent sum and the joint shuffle support treating Latitude and Longitude as separately rankable, or does it suggest something else about how they should be read together?


## 7. Final model reliability assessment

Write a short (4-6 sentence) evidence-based assessment of this model, addressing each point below using evidence from this notebook, not a general impression:

- How well does the model predict? (cite the MAE/R² from Step 2)
- Which features appear important, and by which method? (Step 3)
- What does the local (SHAP) explanation tell us about the one prediction you examined, and what does it *not* tell us? (Steps 4-5)
- What does the correlated-feature check in Step 6 limit your confidence in?
- What additional evidence would increase your confidence in this model's explanations?

**Your response:**

*(write here)*
